# 예제 | 대구 일평균기온 데이터 살펴보기

### 실행 준비
1. 노션에서 코랩을 열고 **파일 → Drive에 사본 저장**을 선택합니다.
2. **런타임 → 모두 실행**을 누르거나 각 셀 왼쪽 ▶를 위에서부터 누릅니다. CPU로 실행합니다.
3. CSV 자동 로딩이 되지 않으면 노션의 Drive CSV를 다운로드하고, 코랩 왼쪽 **파일 → 세션 저장소에 업로드**로 올린 뒤 다시 실행합니다. 파일 이름은 `daegu_143_daily_temperature.csv`로 유지합니다.


## 1. 날짜별 CSV 읽기
2021~2025년 기상청 ASOS 대구(143) 시간자료에서 기온을 날짜별로 집계했습니다. `hours_observed`는 해당 날짜에 실제 기온값이 있는 시간 수입니다.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

filename = "daegu_143_daily_temperature.csv"
paths = [Path(filename), Path("data") / filename]
url = "https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/week11-climate/data/" + filename
source = next((path for path in paths if path.is_file()), url)
raw = pd.read_csv(source)
display(raw.head())
print("관측 날짜 수", len(raw))


## 2. 날짜와 결측 점검
날짜를 정렬하고 달력 기준으로 하루 간격을 맞춥니다. 24시간을 채우지 못한 날은 기온을 결측으로 처리합니다.

In [ ]:
raw["date"] = pd.to_datetime(raw["date"], errors="raise")
assert raw["date"].is_unique, "날짜가 중복되었습니다."
daily = raw.set_index("date").sort_index().asfreq("D")
incomplete = daily["hours_observed"].fillna(0).ne(24)
print("24시간 미만 관측일 수", incomplete.sum())
temp = daily["mean_temp_c"].where(~incomplete)
assert len(temp) == 1826


## 3. 일별 기온과 7일 이동평균 그리기
당일을 포함한 최근 7일을 평균하면 일별 변동이 완화됩니다. 두 곡선의 최고·최저 시점이 어떻게 다른지 확인합니다.

In [ ]:
ax = temp.plot(figsize=(12, 4), alpha=0.45, label="Daily mean")
temp.rolling(7).mean().plot(ax=ax, label="Trailing 7-day mean")
ax.set(ylabel="Temperature (°C)", xlabel="Date")
ax.legend()
plt.show()


### 확인 질문
1. 7일 이동평균이 일별 평균기온보다 완만한 이유를 설명하세요.
2. 24시간 미만 관측일을 왜 0°C로 채우면 안 되는지 적으세요.